In [1]:
import pandas as pd
import numpy as np

In [3]:
payments = pd.read_csv("../Data/payments_dataset.csv")

In [4]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [5]:
payments.describe(include = "all")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
count,103886,103886.000000,103886,103886.000000,103886.000000
unique,99440,NaN,5,NaN,NaN
top,fa65dad1b0e818e3ccc5cb0e39231352,NaN,credit_card,NaN,NaN
freq,29,NaN,76795,NaN,NaN
mean,NaN,1.092679,NaN,2.853349,154.100380
std,NaN,0.706584,NaN,2.687051,217.494064
min,NaN,1.000000,NaN,0.000000,0.000000
25%,NaN,1.000000,NaN,1.000000,56.790000
50%,NaN,1.000000,NaN,1.000000,100.000000
75%,NaN,1.000000,NaN,4.000000,171.837500


In [6]:
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [7]:
payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [8]:
payments.duplicated().sum()

np.int64(0)

In [9]:
payments.duplicated(
    subset=["order_id", "payment_sequential"]
).sum()

np.int64(0)

In [10]:
# payment validation
payments[
    payments["payment_value"] <= 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [11]:
pay_val = payments[payments["payment_value"] <= 0]
len(pay_val)

9

As some payment_value is equal to zero but we are not deleting them as it can disrupt our data. It's not completely 
invalid as payment value can be zero by getting any offer.
No payment value has negative value

In [13]:
payments["payment_type"].value_counts() # keeping not defined value as well

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

**FEATURE ENGINEERING**

In [14]:
payments["payment_category"] = payments["payment_type"].replace({
    "credit_card": "Card",
    "debit_card": "Card",
    "voucher": "Voucher",
    "boleto": "Cash-like",
    "not_defined": "Other"
})

In [15]:
payments["installment_type"] = payments["payment_installments"].apply(
    lambda x: "Single Payment" if x == 1 else "EMI"
)

In [16]:
payments["payment_range"] = pd.cut(
    payments["payment_value"],
    bins=[0,100,300,700,10000],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

In [17]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value,payment_category,installment_type,payment_range
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,Card,EMI,Low
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,Card,Single Payment,Low
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,Card,Single Payment,Low
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,Card,EMI,Medium
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,Card,EMI,Medium


In [18]:
import mysql.connector
from urllib.parse import quote_plus
from sqlalchemy import create_engine
print("Installed Successfully!")

Installed Successfully!


In [19]:
# MySQL credentials
username = "root"
password = quote_plus("Sanket@1024")
host = "localhost"
port = "3306"
database = "E_commerce"
# Create connection
engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

# Upload DataFrame to MySQL
payments.to_sql(
    "payments",
    engine,
    if_exists="replace",
    index=False
)

print("Data uploaded successfully!")

Data uploaded successfully!
